In [1]:
from multiprocessing import get_context
import tensorflow as tf
import time
from decimal import Decimal, ROUND_HALF_UP
from scipy.optimize import brentq

from DNNFM_functions import *
from comp_m_functions import static_factor_obs, cov_e_poet

def round(x):
    return int(Decimal(x).to_integral_value(rounding=ROUND_HALF_UP))

#---------------- Main ----------------

def DNN_FM_main(data, data_factor, architecture=1, const_err_cov=2.5, use_CV_err=False, eval_type='frob'):

    data_dm = data
    data_factor_dm = data_factor
    
    # Obtain optimal tuning parameter based on cross-validation or pre-specified values
    opt = opt_hyper_parameters(data=data, data_F=data_factor, architecture=architecture, const_err_cov=const_err_cov, 
                               use_CV_err=use_CV_err, eval_type=eval_type)
    # Compute DNN-FM based on optimal hyper-parameters
    res_DNN_FM = DNN_FM_core(data=data_dm, data_factor=data_factor_dm, architecture=architecture, opt=opt)

    # c_err_min = DNN_FM_cov_e_cmin(data=data_dm, data_factor=data_factor_dm, DNN_model=res_DNN_FM['neural_net'])

    print(opt['const_err_cov'])
    res_DNN_FM_cov = DNN_FM_cov(data=data_dm, data_factor=data_factor_dm, DNN_model=res_DNN_FM['neural_net'], 
                                c_err_cov=opt['const_err_cov'], check_eig=False)
    
    res_DNN_FM.update(res_DNN_FM_cov)

    return res_DNN_FM


#---------------- Functions ----------------

# Core function for creating Neural Network

def DNN_FM_core(data, data_factor, architecture, opt):

    num_n, num_s = data.shape
    num_f = data_factor.shape[1]

    # Create neural network specifications

    if architecture == 1:
        inter_layer = False

        n_layers = 1
        d_rate = 0.2

        if inter_layer:
            c_temp = 2
        else:
            c_temp = 1

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+c_temp)
        activation_functions = [np.nan] * (n_layers+c_temp)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            hidden_layer_s[mm] = 512
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        if inter_layer:
            dropout_rates_tr[len(dropout_rates_tr)-1] = d_rate
            activation_functions[len(activation_functions)-2] = 'relu'
            activation_functions[len(activation_functions)-1] = 'relu'
        else:
            activation_functions[len(activation_functions)-1] = None

    elif architecture == 2:
        inter_layer = False

        n_layers = 3
        d_rate = 0.1

        if inter_layer:
            c_temp = 2
        else:
            c_temp = 1

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+c_temp)
        activation_functions = [np.nan] * (n_layers+c_temp)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            hidden_layer_s[mm] = 256
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        if inter_layer:
            dropout_rates_tr[len(dropout_rates_tr)-1] = d_rate
            activation_functions[len(activation_functions)-2] = 'relu'
            activation_functions[len(activation_functions)-1] = 'relu'
        else:
            activation_functions[len(activation_functions)-1] = None
        
    elif architecture == 3:
        
        inter_layer = False

        n_layers = 1
        d_rate = 0.0

        if inter_layer:
            c_temp = 2
        else:
            c_temp = 1

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+c_temp)
        activation_functions = [np.nan] * (n_layers+c_temp)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            hidden_layer_s[mm] = 512
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        if inter_layer:
            dropout_rates_tr[len(dropout_rates_tr)-1] = d_rate
            activation_functions[len(activation_functions)-2] = 'relu'
            activation_functions[len(activation_functions)-1] = 'relu'
        else:
            activation_functions[len(activation_functions)-1] = None
    
    elif architecture == 4:

        inter_layer = False

        n_layers = 3
        d_rate = 0.0

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+1)
        activation_functions = [np.nan] * (n_layers+1)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            if mm == 0:
                hidden_layer_s[mm] = 256
            elif mm == 1:
                hidden_layer_s[mm] = 128
            elif mm == 2:
                hidden_layer_s[mm] = 64
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        activation_functions[len(activation_functions)-1] = None
    
    elif architecture == 5:

        inter_layer = False

        n_layers = 3
        d_rate = 0.2

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+1)
        activation_functions = [np.nan] * (n_layers+1)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            if mm == 0:
                hidden_layer_s[mm] = 256
            elif mm == 1:
                hidden_layer_s[mm] = 128
            elif mm == 2:
                hidden_layer_s[mm] = 64
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        activation_functions[len(activation_functions)-1] = None


    elif architecture == 6:

        inter_layer = False

        n_layers = 3
        d_rate = 0.0

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+1)
        activation_functions = [np.nan] * (n_layers+1)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            if mm == 0:
                hidden_layer_s[mm] = 32
            elif mm == 1:
                hidden_layer_s[mm] = 16
            elif mm == 2:
                hidden_layer_s[mm] = 8
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        activation_functions[len(activation_functions)-1] = None

    elif architecture == 7:

        inter_layer = False

        n_layers = 3
        d_rate = 0.2

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+1)
        activation_functions = [np.nan] * (n_layers+1)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            if mm == 0:
                hidden_layer_s[mm] = 32
            elif mm == 1:
                hidden_layer_s[mm] = 16
            elif mm == 2:
                hidden_layer_s[mm] = 8
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        activation_functions[len(activation_functions)-1] = None

    elif architecture == 8:

        inter_layer = False

        n_layers = 3
        d_rate = 0.2

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+1)
        activation_functions = [np.nan] * (n_layers+1)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            if mm == 0:
                hidden_layer_s[mm] = 32
            elif mm == 1:
                hidden_layer_s[mm] = 16
            elif mm == 2:
                hidden_layer_s[mm] = 8
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        activation_functions[len(activation_functions)-1] = None

    elif architecture == 9:

        inter_layer = False

        n_layers = 3
        d_rate = 0.2

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+1)
        activation_functions = [np.nan] * (n_layers+1)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            if mm == 0:
                hidden_layer_s[mm] = 32
            elif mm == 1:
                hidden_layer_s[mm] = 16
            elif mm == 2:
                hidden_layer_s[mm] = 8
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        activation_functions[len(activation_functions)-1] = None

    elif architecture == 10:

        inter_layer = False

        n_layers = 3
        d_rate = 0.2

        hidden_layer_s = [np.nan] * n_layers
        dropout_rates_tr = [np.nan] * (n_layers+1)
        activation_functions = [np.nan] * (n_layers+1)
        dropout_rates_tr[0] = d_rate
        for mm in range(0, n_layers):
            if mm == 0:
                hidden_layer_s[mm] = 256
            elif mm == 1:
                hidden_layer_s[mm] = 128
            elif mm == 2:
                hidden_layer_s[mm] = 64
            dropout_rates_tr[mm+1] = d_rate
            activation_functions[mm] = 'relu'
        
        activation_functions[len(activation_functions)-1] = None


    # Optimization options
    optimizer = 'Adam'
    max_iter = 2000                 # maximum number of iterations
    max_iter_nc = 50                # maximum number of iterations early stopping
    
    split_ratio = 0.3               # split ratio for training and validation set
    batch_s = 256                   # batch size
    use_bias = False                # include bias in neural net, if true

    # Define early stopping criterion
    early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=max_iter_nc, mode='min')

    learning_rate = opt['learning_rate']
    reg_par_w = opt['reg_par_w']          # regularization parameter for weights
    reg_par_b = opt['reg_par_b']          # regularization parameter for bias
    el_r_pro = opt['el_r_pro']            # split between elastic net and lasso: 1 - only l1 norm

    """
    lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
    learning_rate,
    decay_steps=1,
    decay_rate=1,
    staircase=False)
    """

    # Create sparse neural network and compile it
    neural_net = sparse_nn(hidden_layer_s, activation_functions, dropout_rates_tr, num_s, num_f,
                reg_par_w, reg_par_b, el_r_pro, max_iter_nc, max_iter, optimizer, learning_rate, use_bias, inter_layer)

    neural_net.build_neural_network()
    # Compile and fit model
    neural_net.compile_nn()

    fit_nn = neural_net.model.fit(data_factor, data, epochs=max_iter,
                        validation_split=split_ratio, shuffle=True, batch_size=batch_s,
                        callbacks=early_stopping, verbose=0)
    
    # Compute market sensitivity
    with tf.GradientTape() as tape:
        data_factor_ts = tf.convert_to_tensor(data_factor[-1,:].reshape(1,-1), dtype=tf.float32)
        tape.watch(data_factor_ts)
        y_pred = neural_net.model(data_factor_ts)

    market_sens = tape.jacobian(y_pred, data_factor_ts)[-1,:,-1,:].numpy() 

    # Store for analysis
    market_return_t = data_factor_ts.numpy()[-1, :].reshape(1,-1)
    market_sensitivity = (market_sens, market_return_t)

    res = {'neural_net': neural_net, 'opt': opt, 'market_sensitivity': market_sensitivity}
    return res

#-------------------------------------------

def DNN_FM_cov(data, data_factor, DNN_model, c_err_cov, check_eig=False):

    num_n, num_s = data.shape

    y_hat_nn = DNN_model.model(data_factor).numpy()
    resd_nn = data - y_hat_nn

    sig_hat_e = thres_resd_new(resd_nn, c_err_cov, num_s, num_n)
    cov_f_nnet = np.cov(y_hat_nn.T)

    if not check_eig:
                
        sigma_y_nnet = cov_f_nnet + sig_hat_e
    else:

        cond = True
        const_c = 0
        while cond:

            sig_hat_e = thres_resd_new(resd_nn, c_err_cov+const_c, num_s, num_n)

            cond = (round(min(np.linalg.eig(sig_hat_e)[0]),2) < 0.01) or (np.linalg.cond(sig_hat_e) > num_s*10)
            const_c+=0.01
        
        sigma_y_nnet = cov_f_nnet + sig_hat_e

    inv_sigma_y_nnet = sig_inv_f_nnet(cov_f_nnet, sig_hat_e)

    res = {'sigma_hat': sigma_y_nnet, 'sigma_f_hat': cov_f_nnet, 'sigma_e_hat': sig_hat_e, 
           'inv_sigma_hat': inv_sigma_y_nnet}

    return res

def DNN_FM_cov_e_cmin(data, data_factor, DNN_model):

    num_n, num_s = data.shape

    y_hat_nn = DNN_model.model(data_factor).numpy()
    resd_nn = data - y_hat_nn

    num_n, num_s = resd_nn.shape

    f = lambda c: mineig_cov_e(resd=resd_nn, c_err_cov=c, num_s=num_s, num_n=num_n)

    if (f(50) * f(-50) < 0):
        r = brentq(f, -50, 50)
        return max(0, r)
    else:
        c = 0
        return c

def mineig_cov_e(resd, c_err_cov, num_s, num_n):

    return min(np.linalg.eig(thres_resd_new(resd, c_err_cov, num_s, num_n))[0])


#-------------------------------------------

# Function for determining optimal regularization and learning rate based on cross-validation or fixed
def opt_hyper_parameters(data, data_F, architecture, const_err_cov, use_CV_err, eval_type):
    
    parallel = False

    if architecture == 1 or architecture == 5 or architecture == 7:
        reg_par_w = 0.0005           # regularization parameter for weights
        reg_par_b = 0.0005           # regularization parameter for bias
    elif architecture == 9 or architecture == 10:
        reg_par_w = 0.005           # regularization parameter for weights
        reg_par_b = 0.005           # regularization parameter for bias
    else:
        reg_par_w = 0.0             # regularization parameter for weights
        reg_par_b = 0.0             # regularization parameter for bias
    el_r_pro = 1                    # split between elastic net and lasso: 1 - only l1 norm
    # Alternative specification for the learning rate
    learning_rate = 0.0005

    if use_CV_err:

        range_cov_err = np.arange(0, const_err_cov+0.6, 0.1)

        """
        opt = block_cv(data=data, data_F=data_F, architecture=architecture, 
                       reg_par=reg_par_w, lr=learning_rate, range_cov_err=range_cov_err, 
                       eval_type=eval_type, test_size=10, parallel=parallel)
        """
        
        opt = cv_split(data=data, data_F=data_F, architecture=architecture, 
                       reg_par=reg_par_w, lr=learning_rate, range_cov_err=range_cov_err, 
                       eval_type=eval_type)

    else:

        opt = {'learning_rate': learning_rate, 'reg_par_w': reg_par_w, 
            'reg_par_b': reg_par_b, 'el_r_pro': el_r_pro, 'const_err_cov': const_err_cov}
    
    return opt
        
#-------------------------------------------

# Function that determines hyperparameters based on block cross-validation

def block_cv(data, data_F, architecture, reg_par, lr, range_cov_err, eval_type, min_train_ratio=0.8, test_size=5, parallel=False):

    start = time.time()

    T, p = data.shape
    train_size = int(np.floor(min_train_ratio * T)) # Get training size
    numBlocks = int(np.floor((T - train_size) / test_size)) # Compute number of blocks
    
    # Show error message if test size is too large
    assert numBlocks > 1, f"Test size is too large"

    # Predefine dictionary for options
    opt = {'learning_rate': lr, 'reg_par_w': reg_par, 'reg_par_b': reg_par, 'el_r_pro': 1}
    
    res_mat = np.empty((numBlocks,len(range_cov_err)))
    res_mat[:] = np.nan

    for bl in range(1, numBlocks+1):
        idxTrainX = list(range((bl-1)*test_size,train_size+(bl-1)*test_size))

        if bl == numBlocks:
            idxTestX = list(range(train_size+(bl-1)*test_size,T))
        else:
            idxTestX = list(range(train_size+(bl-1)*test_size,train_size+bl*test_size))

        
        data_ntrain, data_mean, data_std = normalize_dat(data[idxTrainX,:])
        data_F_ntrain, data_F_mean, data_F_std = normalize_dat(data_F[idxTrainX,:])    

        test_res = np.empty((len(idxTestX),len(range_cov_err)))
        test_res[:] = np.nan
        
        # if bl == 1:
        res_DNN_FM = DNN_FM_core(data_ntrain, data_F_ntrain, architecture, opt)
        neural_net = res_DNN_FM['neural_net']


        for idx_t in range(1, len(idxTestX)+1):

            ind_test_temp = idxTrainX[idx_t:]+idxTestX[:idx_t]

            data_ntest, data_mean, data_std = normalize_dat(data[ind_test_temp,:])
            data_F_ntest, data_F_mean, data_F_std = normalize_dat(data_F[ind_test_temp,:])

            y_hat = neural_net.model(data_F_ntrain).numpy()
            resd_nn = data_ntrain - y_hat

            num_n, num_s = resd_nn.shape

            sigma_test = cov_sfm(data_ntest - neural_net.model(data_F_ntest).numpy())

            """
            # define the number of worker processes to use
            num_processes = 2

            combined_list = [(resd_nn, sigma_test, eval_type, const_err_item) for const_err_item in range_cov_err]

            if parallel:
                # create a pool of worker processes
                with get_context("spawn").Pool(processes=num_processes) as pool:
                    ret = pool.imap(err_cv_core, combined_list)
                    pool.close()
                    pool.join()
            else:
                ret = map(err_cv_core, combined_list)

            for indx, err_measure in enumerate(ret):
                test_res[idx_t-1,indx] = err_measure
            """

            # sigma_test = cov_sfm(data_ntest - neural_net.model(data_F_ntest).numpy())

            rate_thres = np.sqrt((np.log(num_s))/num_n)
            sig_e_samp = np.cov(resd_nn.T)
            thet_par = np.empty((num_s, num_s))
            thet_par[:] = np.nan

            for ii in range(0, num_s):
                for jj in range(0, num_s):
                    thet_par[ii, jj] = np.mean(np.abs(resd_nn[:, ii] * resd_nn[:, jj] - sig_e_samp[ii, jj]))

            sig_e_diag = np.diag(sig_e_samp)
            
            """
            sig_e_diag = np.diag(np.diag(sig_e_samp)**(0.5))
            R = np.linalg.inv(sig_e_diag) @ sig_e_samp @ np.linalg.inv(sig_e_diag)
            """
            
            for c_idx in range(0,len(range_cov_err)):
                lam = rate_thres * range_cov_err[c_idx] * thet_par
                
                """
                M = soft_t(R, lam)
                M = M - np.diag(np.diag(M)) + np.eye(num_s)
                sig_hat_e = sig_e_diag @ M @ sig_e_diag
                """
                
                sig_hat_e = soft_t(sig_e_samp, lam)
                np.fill_diagonal(sig_hat_e, sig_e_diag)
                
                # sig_hat_e = thres_resd_new(resd_nn, range_cov_err[c_idx], num_s, num_n)

                if min(np.linalg.eig(sig_hat_e)[0]) < 0:
                    test_res[idx_t-1,c_idx] = np.inf
                else:
                    if eval_type == 'frob':
                        test_res[idx_t-1,c_idx] = np.linalg.norm(sig_hat_e - sigma_test, ord='fro')**2
                    elif eval_type == 'spec':
                        test_res[idx_t-1,c_idx] = np.linalg.norm(sig_hat_e - sigma_test, ord=2)**2

            
        res_mat[bl-1,:] = np.mean(test_res, axis=0)


     
    idx_opt = np.where(res_mat.mean(axis=0) == np.nanmin(res_mat.mean(axis=0)))

    opt.update({'const_err_cov': range_cov_err[idx_opt][0]})
    end = time.time()

    print(end - start)
    return opt

def err_cv_core(tuple_in):

    resd_nn, sigma_test, eval_type, const_err_cov = tuple_in[0], tuple_in[1], tuple_in[2], tuple_in[3]

    num_n, num_s = resd_nn.shape

    st = time.time()
    sig_hat_e = thres_resd_new(resd_nn, const_err_cov, num_s, num_n)
    print(time.time()-st)

    if min(np.linalg.eig(sig_hat_e)[0]) < 0:
        test_res = np.inf
    else:
        if eval_type == 'frob':
            test_res = np.linalg.norm(sig_hat_e - sigma_test, ord='fro')
        elif eval_type == 'spec':
            st1 = time.time()
            test_res = np.linalg.norm(sig_hat_e - sigma_test, ord=2)
            print(time.time()-st1)

    return test_res

def cv_split(data, data_F, architecture, reg_par, lr, range_cov_err, eval_type):

    start = time.time()

    # Predefine dictionary for options
    opt = {'learning_rate': lr, 'reg_par_w': reg_par, 'reg_par_b': reg_par, 'el_r_pro': 1}

    data_dm, _, _ = normalize_dat_sim(data)
    data_F_dm, _, _ = normalize_dat_sim(data_F)    

    res_DNN_FM = DNN_FM_core(data_dm, data_F_dm, architecture, opt)
    neural_net = res_DNN_FM['neural_net']

    y_hat = neural_net.model(data_F_dm).numpy()
    resd_nn = data_dm - y_hat

    n_folds = 10

    res_mat = np.empty((n_folds,len(range_cov_err)))
    res_mat[:] = np.nan

    split_sample, _ = ts_train_test_split(resd_nn, n_folds, train_size=0.5)

    for m_idx in range(0, n_folds):


            resd_nn_s1 = split_sample[m_idx][0]
            resd_nn_s2 = split_sample[m_idx][1]

            num_n, num_s = resd_nn_s1.shape

            sigma_test = cov_sfm(resd_nn_s2) # np.cov(resd_nn_s2.T) # 

            sig_e_samp, thet_par = thres_cov_resd_aux(resd_nn_s1, num_s)

            for c_idx in range(0,len(range_cov_err)):
                
                sig_hat_e = thres_cov_resd(sig_e_samp, thet_par, range_cov_err[c_idx], num_s, num_n)

                if min(np.linalg.eig(sig_hat_e)[0]) < 0:
                    res_mat[m_idx,c_idx] = np.inf
                else:
                    if eval_type == 'frob':
                        res_mat[m_idx,c_idx] = np.linalg.norm(sig_hat_e - sigma_test, ord='fro')**2
                    elif eval_type == 'spec':
                        res_mat[m_idx,c_idx] = np.linalg.norm(sig_hat_e - sigma_test, ord=2)**2

    idx_opt = np.where(res_mat.mean(axis=0) == np.nanmin(res_mat.mean(axis=0)))

    opt.update({'const_err_cov': range_cov_err[idx_opt][0]})
    end = time.time()

    print(end - start)

    return opt

def ts_train_test_split(X, n_folds = 5, train_size=0.5):

    test_size = 1 - train_size
    n_obs = X.shape[0]

    size_split = n_obs - n_folds

    n_train = round(size_split * train_size)
    n_test = round(size_split * test_size)

    split_t = (list(range(0, n_train)), list(range(n_train, n_train+n_test)))

    split_sample = []
    split_index = []

    for jj in range(0, n_folds):
        split_index.append(([el1 + jj for el1 in split_t[0]], [el2 + jj for el2 in split_t[1]]))
        split_sample.append((X[split_index[jj][0],:], X[split_index[jj][1],:]))

    return split_sample, split_index


def gmv_weights(Theta_hat):
    """
    Compute Global Minimum Variance (GMV) portfolio weights (Section 6.1).
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # w* = (Θ 1_p) / (1_p' Θ 1_p)
    numerator = Theta_hat @ ones_p
    denominator = ones_p @ Theta_hat @ ones_p
    
    if np.abs(denominator) < 1e-10:
        # Fallback to equal weights if precision matrix is near-singular
        return ones_p / p
    
    w_star = numerator / denominator
    
    return w_star

def mv_weights(Theta_hat, mu, target_variance=0.15**2/12):
    """
    Compute Mean-Variance portfolio weights to maximize return given a target variance.
    
    Solves the constrained optimization:
    max w' mu  subject to  w' Sigma w <= target_variance  and  w' 1 = 1
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected returns
    target_variance : float
        Target maximum portfolio variance
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute key quantities
    A = ones_p @ Theta_hat @ ones_p  # 1' Theta 1
    B = ones_p @ Theta_hat @ mu      # 1' Theta mu  
    C = mu @ Theta_hat @ mu          # mu' Theta mu
    D = A * C - B * B                # Determinant
    
    # Check for singularity
    if np.abs(D) < 1e-10 or np.abs(A) < 1e-10:
        print('SINGULARITY: Falling back to equal weights')
        return ones_p / p
        
    # Minimum possible variance (Global Minimum Variance)
    min_variance = 1.0 / A
    
    # Check feasibility
    if target_variance < min_variance:
        print(f'INFEASIBLE: Target variance {target_variance:.6f} is below GMV variance {min_variance:.6f}. Returning GMV portfolio.')
        return (Theta_hat @ ones_p) / A

    # Map target variance to maximum expected return on the efficient frontier
    # The efficient frontier equation is: variance = (A*mu^2 - 2*B*mu + C) / D
    # Solving for the upper half (max return) yields:
    target_return = (B + np.sqrt(D * (A * target_variance - 1))) / A
    
    # Compute Lagrange multipliers using the derived target_return
    lambda1 = (C - B * target_return) / D
    lambda2 = (A * target_return - B) / D
    
    # Compute weights: w = lambda1 * Theta^{-1} 1 + lambda2 * Theta^{-1} mu
    w_star = lambda1 * (Theta_hat @ ones_p) + lambda2 * (Theta_hat @ mu)
    
    return w_star

def msr_weights(Theta_hat, mu):
    """
    Compute Maximum Sharpe Ratio portfolio weights.
    
    The maximum Sharpe ratio portfolio solves:
    max (w' mu) / sqrt(w' Sigma w)
    
    Solution (when mu represents excess returns):
    w ∝ Sigma^{-1} mu = Theta mu
    
    Then normalize so that sum(w) = 1.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected excess returns
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights (sum to 1)
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute unnormalized weights: w ∝ Theta mu
    w_unnorm = Theta_hat @ mu
    
    # Normalize to sum to 1
    weight_sum = np.sum(w_unnorm)
    
    if np.abs(weight_sum) < 1e-10:
        print('WARNING: Weight sum near zero, returning equal weights')
        return ones_p / p
    
    w_star = w_unnorm / weight_sum
    
    return w_star

2026-11-06 22:15:11.325172: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def train_random_forest(df, train_start, train_end, features=['mom12m', 'mve', 'bm'], clip_tails=True):
    """
    Train Random Forest Regressor on historical data to predict return magnitude.
    """
    train_df = df[(df['datadate'] >= train_start) & (df['datadate'] <= train_end)].copy()
    
    # Drop rows missing features or the continuous target
    train_df = train_df.dropna(subset=features + ['ret_fwd_1'])
    
    # Winsorize (clip) extreme outliers to protect the Regressor's MSE loss function
    if clip_tails:
        lower_cap = train_df['ret_fwd_1'].quantile(0.01)
        upper_cap = train_df['ret_fwd_1'].quantile(0.99)
        train_df['target'] = train_df['ret_fwd_1'].clip(lower_cap, upper_cap)
    else:
        train_df['target'] = train_df['ret_fwd_1']
    
    X_train = train_df[features]
    y_train = train_df['target']
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Use Regressor instead of Classifier
    rf_model = RandomForestRegressor(
        n_estimators=200,      # More trees to smooth variance
        max_depth=8,           # Increased from 5 to allow deeper interactions
        min_samples_leaf=15,   # Decreased from 50 to allow smaller, more specific leaf nodes
        max_features='sqrt',   # Forces trees to use different features, increasing variance
        random_state=42, 
        n_jobs=-1
    )
    rf_model.fit(X_train_scaled, y_train)
    
    print(f"  Training samples: {len(train_df)}")
    print(f"  Training R-squared: {rf_model.score(X_train_scaled, y_train):.4f}")
    
    return rf_model, scaler


def select_stocks_rf_quantiles(df, predict_date, rf_model, scaler, 
                                         features=['mom12m', 'mve', 'bm'],
                                         top_q=0.90, bottom_q=0.10):
    """
    Select top and bottom percentiles of stocks based on predicted expected return.
    """
    predict_df = df[df['datadate'] == predict_date].copy()
    predict_df = predict_df.dropna(subset=features)
    
    if len(predict_df) == 0:
        print(f"  ⚠ No stocks with complete data on {predict_date}")
        return []
    
    X_predict = predict_df[features]
    X_predict_scaled = scaler.transform(X_predict)
    
    # Regressor uses .predict() to output a continuous return estimate, not a probability
    predict_df['expected_return'] = rf_model.predict(X_predict_scaled)
    
    # Calculate quantiles on the predicted magnitude
    upper_bound = predict_df['expected_return'].quantile(top_q)
    lower_bound = predict_df['expected_return'].quantile(bottom_q)
    
    # Screen out the dead zone
    selected_df = predict_df[
        (predict_df['expected_return'] >= upper_bound) | 
        (predict_df['expected_return'] <= lower_bound)
    ]
    
    selected_permnos = selected_df['permno'].tolist()
    
    print(f"  Stocks evaluated: {len(predict_df)}")
    print(f"  Upper expected return bound (Top {100-top_q*100:.0f}%): {upper_bound:.4f}")
    print(f"  Lower expected return bound (Bottom {bottom_q*100:.0f}%): {lower_bound:.4f}")
    print(f"  Stocks selected: {len(selected_permnos)}")
    
    return selected_permnos

In [3]:
def integrated_backtest(df_daily, df_monthly,
                              test_start_date='2020-01-31', 
                              test_end_date='2024-04-30',
                              rf_train_years=15,
                              features=['mom12m', 'mve', 'bm'],
                              top_q=0.90,
                              bottom_q=0.10,
                              lookback_window=60, # 60 daily periods ~ 3 months
                              transaction_cost=0.001, # Adjusted for daily trading
                              data_factor=None,
                              dnn_architecture=5,
                              dnn_const_err_cov=2.5,
                              portfolio_type='all',  
                              mv_target_variance=0.15**2/252,
                              verbose=True):
    """
    Integrated daily backtest combining monthly Random Forest stock selection 
    with daily DNN-FM portfolio optimization.
    """
    df_daily = df_daily.copy()
    df_monthly = df_monthly.copy()
    
    df_daily['datadate'] = pd.to_datetime(df_daily['datadate'])
    df_monthly['datadate'] = pd.to_datetime(df_monthly['datadate'])
    
    if data_factor is not None:
        data_factor.index = pd.to_datetime(data_factor.index)
        
    daily_dates = sorted(df_daily['datadate'].unique())
    
    # Parse test dates
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    try:
        test_start_idx = daily_dates.index(test_start_dt)
        test_end_idx = daily_dates.index(test_end_dt)
    except ValueError as e:
        raise ValueError(f"Date not found in daily DataFrame: {e}")
    
    if test_start_idx < lookback_window:
        raise ValueError(f"Not enough daily data for lookback window.")
    
    if portfolio_type == 'all':
        portfolio_types = ['gmv', 'mv', 'msr']
    else:
        portfolio_types = [portfolio_type]
    
    # Storage for each portfolio type
    results_storage = {ptype: {
        'returns': [], 'dates': [], 'weights_list': [],
        'turnover_list': [], 'gross_returns': [],
        'prev_weights_dict': {}, 'prev_oos_returns_dict': {},
        'prev_gross_return': 0.0
    } for ptype in portfolio_types}
    
    # Cache to store RF selected stocks so we only train once per month
    rf_universe_cache = {}
    
    if verbose:
        print("="*70)
        print("DAILY INTEGRATED BACKTEST: RF SELECTION + DNN-FM")
        print(f"Test Period: {test_start_date} to {test_end_date}")
        print("="*70)
    
    # Daily loop
    for t in range(test_start_idx, test_end_idx + 1):
        current_date = daily_dates[t]
        
        # --- 1. MONTHLY RF STOCK SELECTION ---
        # Calculate the previous month (T-1) for safe, lookahead-free signal lookup
        prev_month_period = (current_date - pd.DateOffset(months=1)).to_period('M')
        
        if prev_month_period not in rf_universe_cache:
            month_mask = df_monthly['datadate'].dt.to_period('M') == prev_month_period
            if not month_mask.any():
                if verbose:
                    print(f"\n⚠ No monthly data found for {prev_month_period}, skipping month")
                rf_universe_cache[prev_month_period] = []
            else:
                retrain_date = df_monthly[month_mask]['datadate'].max()
                
                if verbose:
                    print(f"\n{'='*70}")
                    print(f"TRAINING RF FOR TRADING MONTH: {current_date.strftime('%B %Y')}")
                    print(f"Using monthly features at: {retrain_date.strftime('%Y-%m-%d')}")
                    print(f"{'='*70}")
                
                train_end = retrain_date - pd.DateOffset(months=1)
                train_start = retrain_date - pd.DateOffset(years=rf_train_years)
                
                rf_model, scaler = train_random_forest(
                    df_monthly, train_start, train_end, features=features
                )
                
                current_permnos = select_stocks_rf_quantiles(
                    df_monthly, retrain_date, rf_model, scaler=scaler,
                    features=features, top_q=top_q, bottom_q=bottom_q
                )
                rf_universe_cache[prev_month_period] = current_permnos
                
        current_permnos = rf_universe_cache[prev_month_period]
        
        if len(current_permnos) == 0:
            if verbose:
                print(f"[{current_date.strftime('%Y-%m-%d')}] ⚠ No stocks in RF universe, skipping")
            continue

        # --- 2. DAILY DNN-FM MODEL ---
        window_start_date = daily_dates[t - lookback_window]
        window_end_date = daily_dates[t - 1]
        
        # Filter daily data to RF selected stocks only
        train_data = df_daily[
            (df_daily['datadate'] >= window_start_date) & 
            (df_daily['datadate'] <= window_end_date) &
            (df_daily['permno'].isin(current_permnos))
        ]
        
        returns_pivot = train_data.pivot(index='datadate', columns='permno', values='ret_fwd_1')
        window_dates = daily_dates[t - lookback_window : t]
        returns_pivot = returns_pivot.reindex(index=window_dates)
        
        # =========================================================
        # EXACT PRESERVATION OF YOUR ORIGINAL FACTOR LOGIC
        # =========================================================
        train_factor = data_factor.loc[window_start_date : window_end_date]
        # =========================================================
        
        nan_assets = returns_pivot.columns[returns_pivot.isna().any()]
        filtered_pivot = returns_pivot.drop(columns=nan_assets)
        
        current_assets = filtered_pivot.columns.tolist()
        Y = filtered_pivot.values
        n_train, p_current = Y.shape
        
        if verbose:
            print(f"\n[{current_date.strftime('%Y-%m-%d')}] | Assets: {p_current}/{len(current_permnos)} valid")
        
        if n_train < lookback_window or p_current < 2:
            if verbose:
                print(f"  ⚠ Insufficient daily data, using previous weights")
            new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                               for ptype in portfolio_types}
        else:
            try:
                Y_bar = Y.mean(axis=0)
                Y_star = Y - Y_bar
                
                # =========================================================
                # EXACT PRESERVATION OF YOUR ORIGINAL DNN-FM LOGIC
                # =========================================================
                if verbose:
                    print(f"  Running DNN-FM...")
                F = train_factor.values.astype(float)
                res_nnet_fm = DNN_FM_main(Y_star, F, architecture=dnn_architecture, 
                                         const_err_cov=dnn_const_err_cov, 
                                         use_CV_err=False, eval_type='frob')
                Theta_hat = res_nnet_fm['inv_sigma_hat']
                # =========================================================
                
                new_weights_dict = {}
                
                if 'gmv' in portfolio_types:
                    w_gmv = gmv_weights(Theta_hat)
                    new_weights_dict['gmv'] = {asset: w_gmv[i] for i, asset in enumerate(current_assets)}
                
                if 'mv' in portfolio_types or 'msr' in portfolio_types:
                    mu = Y_bar
                    if 'mv' in portfolio_types:
                        w_mv = mv_weights(Theta_hat, mu, target_variance=mv_target_variance)
                        new_weights_dict['mv'] = {asset: w_mv[i] for i, asset in enumerate(current_assets)}
                    
                    if 'msr' in portfolio_types:
                        w_msr = msr_weights(Theta_hat, mu)
                        new_weights_dict['msr'] = {asset: w_msr[i] for i, asset in enumerate(current_assets)}
                        
            except Exception as e:
                if verbose:
                    print(f"  ✗ Error: {e}")
                new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                                   for ptype in portfolio_types}
        
        # --- 3. OOS RETURNS & TRANSACTION COSTS ---
        oos_data = df_daily[df_daily['datadate'] == current_date]
        oos_returns_series = oos_data.set_index('permno')['ret_fwd_1'].dropna()
        oos_returns_dict = oos_returns_series.to_dict()
        
        for ptype in portfolio_types:
            weights = new_weights_dict[ptype]
            weight_sum = sum(weights.values())
            if weight_sum > 1e-10:
                weights = {k: v/weight_sum for k, v in weights.items()}
            else:
                weights = results_storage[ptype]['prev_weights_dict'].copy()
                
            prev_weights = results_storage[ptype]['prev_weights_dict']
            prev_oos_returns = results_storage[ptype]['prev_oos_returns_dict']
            prev_gross_ret = results_storage[ptype]['prev_gross_return']
            
            common_assets = set(weights.keys()) & set(oos_returns_dict.keys())
            
            if len(common_assets) == 0:
                continue
            
            common_weights = {a: weights[a] for a in common_assets}
            common_weight_sum = sum(common_weights.values())
            if common_weight_sum > 1e-10:
                common_weights = {k: v/common_weight_sum for k, v in common_weights.items()}
            else:
                continue
            
            gross_return = sum(common_weights[a] * oos_returns_dict[a] for a in common_assets)
            if np.isnan(gross_return) or np.isinf(gross_return):
                continue
            
            # Daily Transaction Costs
            if len(prev_weights) > 0:
                adjusted_prev = {}
                for asset, prev_w in prev_weights.items():
                    if asset in prev_oos_returns:
                        prev_r = prev_oos_returns[asset]
                        if abs(1 + prev_gross_ret) > 1e-6:
                            adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_ret)
                        else:
                            adjusted_prev[asset] = 0.0
                    else:
                        if abs(1 + prev_gross_ret) > 1e-6:
                            adjusted_prev[asset] = prev_w / (1 + prev_gross_ret)
                        else:
                            adjusted_prev[asset] = 0.0
                
                all_assets = set(adjusted_prev.keys()) | set(common_weights.keys())
                turnover = sum(abs(common_weights.get(a, 0.0) - adjusted_prev.get(a, 0.0)) for a in all_assets)
                tc = transaction_cost * (1 + gross_return) * turnover
            else:
                turnover = sum(abs(w) for w in common_weights.values())
                tc = transaction_cost * (1 + gross_return) * turnover
            
            net_return = gross_return - tc
            
            results_storage[ptype]['returns'].append(net_return)
            results_storage[ptype]['dates'].append(current_date)
            results_storage[ptype]['weights_list'].append(common_weights.copy())
            results_storage[ptype]['turnover_list'].append(turnover)
            results_storage[ptype]['gross_returns'].append(gross_return)
            
            results_storage[ptype]['prev_weights_dict'] = common_weights.copy()
            results_storage[ptype]['prev_oos_returns_dict'] = {a: oos_returns_dict[a] for a in common_assets}
            results_storage[ptype]['prev_gross_return'] = gross_return
        
        if verbose:
            for ptype in portfolio_types:
                if len(results_storage[ptype]['returns']) > 0:
                    last_idx = len(results_storage[ptype]['returns']) - 1
                    gross_ret = results_storage[ptype]['gross_returns'][last_idx]
                    net_ret = results_storage[ptype]['returns'][last_idx]
                    to = results_storage[ptype]['turnover_list'][last_idx]
                    tc = gross_ret - net_ret
                    print(f"  {ptype.upper()}: Gross={gross_ret:>7.4f} | TO={to:>5.3f} | TC={tc:>7.5f} | Net={net_ret:>7.4f}")
    
    if verbose:
        print("\n" + "="*70)
        print("BACKTEST COMPLETE")
        print("="*70)
    
    results_dict = {}
    
    for ptype in portfolio_types:
        portfolio_returns = results_storage[ptype]['returns']
        
        if len(portfolio_returns) == 0:
            results_dict[ptype] = {'results_df': pd.DataFrame(), 'metrics': {}}
            continue
        
        results_df = pd.DataFrame({
            'date': results_storage[ptype]['dates'],
            'portfolio_return': portfolio_returns,
            'portfolio_gross_return': results_storage[ptype]['gross_returns'],
            'portfolio_turnover': results_storage[ptype]['turnover_list']
        })
        results_df['cumulative_return'] = (1 + results_df['portfolio_return']).cumprod() - 1
        
        mean_return = np.mean(portfolio_returns)
        variance = np.var(portfolio_returns, ddof=1)
        sharpe_ratio = mean_return / np.sqrt(variance) if variance > 0 else 0
        
        # CHANGED: Annualized metrics using 252 for daily trading
        annual_return = mean_return * 252
        annual_volatility = np.sqrt(variance * 252)
        annual_sharpe = annual_return / annual_volatility if annual_volatility > 0 else 0
        
        overall_metrics = {
            'mean_return': mean_return,
            'variance': variance,
            'sharpe_ratio': sharpe_ratio,
            'annual_return': annual_return,
            'annual_volatility': annual_volatility,
            'annual_sharpe_ratio': annual_sharpe,
            'total_return': results_df['cumulative_return'].iloc[-1],
            'avg_turnover': np.mean(results_storage[ptype]['turnover_list']),
            'n_periods': len(portfolio_returns)
        }
        
        results_dict[ptype] = {'results_df': results_df, 'metrics': overall_metrics}
    
    return results_dict

In [4]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = (df.groupby('permno')['ret_excess'].shift(-1) )

In [5]:
df_daily = pd.read_csv('../Paper revisions/filtered_daily_data_w_ticker.csv')
df_daily['ret_fwd_1'] = df_daily.groupby('permno')['ret_excess'].shift(-1)
df_daily = df_daily.rename(columns={'date': 'datadate'})

In [6]:
variables = ['agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc',
            'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 'cash',
            'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho',
            'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 'chpmia',
            'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi',
            'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 'gma', 'grcapx',
            'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 'invest',
            'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 'mve_ia', 'nanalyst',
            'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr',
            'pchgm_pchsale', 'pchsale_pchinvt', 'pchsale_pchrect',
            'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps',
            'rd', 'rd_mve', 'rd_sale', 'realestate', 'retvol', 'roaq',
            'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec',
            'secured', 'securedind', 'sfe', 'sgr', 'sin', 'sp', 'std_dolvol',
            'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']

In [7]:
data_f = pd.read_csv('F-F_Research_Data_Factors_daily.csv')
date_strings = data_f['Date'].astype(str).str.strip().str.replace('.0', '', regex=False)
data_f['Date'] = pd.to_datetime(date_strings, format="%Y%m%d")
data_f = data_f.set_index('Date')
data_f = data_f[['Mkt-RF', 'SMB', 'HML', 'RF']].astype(float)

results = integrated_backtest(
    df_daily = df_daily,
    df_monthly = df,
    test_start_date='2024-02-01',
    test_end_date='2024-04-30',
    rf_train_years=15,
    features=['mom12m', 'mve', 'bm'],
    top_q=0.90,
    bottom_q=0.10,
    lookback_window=180,
    transaction_cost=0.001,
    data_factor=data_f,
    verbose=True
)

# Access results for each portfolio
gmv_results = results['gmv']['results_df']
gmv_metrics = results['gmv']['metrics']

mv_results = results['mv']['results_df']
mv_metrics = results['mv']['metrics']

msr_results = results['msr']['results_df']
msr_metrics = results['msr']['metrics']

# Compare Sharpe ratios
print(f"GMV Sharpe: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MV Sharpe:  {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MSR Sharpe: {msr_metrics['annual_sharpe_ratio']:.4f}")

DAILY INTEGRATED BACKTEST: RF SELECTION + DNN-FM
Test Period: 2024-02-01 to 2024-04-30

TRAINING RF FOR TRADING MONTH: February 2024
Using monthly features at: 2024-01-31
  Training samples: 89406
  Training R-squared: 0.0089
  Stocks evaluated: 500
  Upper expected return bound (Top 10%): 0.0142
  Lower expected return bound (Bottom 10%): 0.0112
  Stocks selected: 100

[2024-02-01] | Assets: 36/100 valid
  Running DNN-FM...
2.5
  GMV: Gross=-0.0027 | TO=1.062 | TC=0.00106 | Net=-0.0037
  MV: Gross= 0.0008 | TO=2.811 | TC=0.00281 | Net=-0.0020
  MSR: Gross= 0.0001 | TO=2.306 | TC=0.00231 | Net=-0.0022

[2024-02-02] | Assets: 36/100 valid
  Running DNN-FM...
2.5
  GMV: Gross=-0.0033 | TO=0.035 | TC=0.00003 | Net=-0.0034
  MV: Gross=-0.0036 | TO=0.269 | TC=0.00027 | Net=-0.0038
  MSR: Gross=-0.0035 | TO=0.230 | TC=0.00023 | Net=-0.0037

[2024-02-05] | Assets: 36/100 valid
  Running DNN-FM...
2.5
  GMV: Gross= 0.0006 | TO=0.031 | TC=0.00003 | Net= 0.0005
  MV: Gross=-0.0094 | TO=0.272 | T

In [9]:
print(f"\n GMV")
print(f"Annualized Sharpe Ratio: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {gmv_metrics['mean_return']*252:.4f}")
print(f"Variance: {gmv_metrics['variance']*252:.4f}")
print(f"Avg Turnover: {gmv_metrics['avg_turnover']:.4f}")

print(f"\n MV")
print(f"Annualized Sharpe Ratio: {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {mv_metrics['mean_return']*252:.4f}")
print(f"Variance: {mv_metrics['variance']*252:.4f}")
print(f"Avg Turnover: {mv_metrics['avg_turnover']:.4f}")

print(f"\n MSR")
print(f"Annualized Sharpe Ratio: {msr_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {msr_metrics['mean_return']*252:.4f}")
print(f"Variance: {msr_metrics['variance']*252:.4f}")
print(f"Avg Turnover: {msr_metrics['avg_turnover']:.4f}")


 GMV
Annualized Sharpe Ratio: 0.9547
Mean Return: 0.0921
Variance: 0.0093
Avg Turnover: 0.1219

 MV
Annualized Sharpe Ratio: 1.2493
Mean Return: 0.2426
Variance: 0.0377
Avg Turnover: 0.5653

 MSR
Annualized Sharpe Ratio: 0.2690
Mean Return: 0.0536
Variance: 0.0398
Avg Turnover: 0.5495
